Initialization

In [1]:
%matplotlib inline
import os
import sys

import time
import csv
import json
import numpy as np 

from src.biology import * 
from src.human import *
from src.evolution import *
from src.physics import *
from src.traditional import * 
    
from src.testing.continous.test_functions import sphere, rastrigin, rosenbrock, griewank, ackley
from src.testing.discrete_problems.TSP           import TSP, TSPSolver
from src.testing.discrete_problems.Knapsack      import KnapsackProblem, KnapsackSolver
from src.testing.discrete_problems.GraphColoring import GraphColoring, GraphColoringSolver

from src.utils.logger import *
from src.benchmark import BenchmarkRunner
from src.visualization.visualize import (
    plot_convergence, plot_box_scores, plot_time_comparison,
    plot_heatmap_ranking, plot_summary_table, plot_overall_comparison,
    plot_discrete_convergence, plot_discrete_bar,
    plot_pathfinding_grids, plot_pathfinding_metrics, plot_pathfinding_summary_table,
    plot_tsp_tour, plot_knapsack_items, plot_graph_coloring,
    plot_scalability, plot_exploration_exploitation
)

SETTING PARAMETERS AND ALGORITHMS

In [2]:
# Parameters Configuration
n_trials = 10
dim = 10
pop_size = 30
max_iter_pop = 1000
max_iter_single = 20000
seed = 42

# Initialize Algorithms
pso = PSO(n_particles=pop_size, max_iter=max_iter_pop, c1=1.49618, c2=1.49618, w=0.7298)
cs = CS(n_nests=pop_size, max_iter=max_iter_pop, pa=0.25)
fa = FA(n_fireflies=pop_size, max_iter=max_iter_pop, alpha=0.5, beta0=1.0, gamma=0.1)
de = DE(pop_size=pop_size, max_iter=max_iter_pop, F=0.8, CR=0.9)
tlbo = TLBO(pop_size=pop_size, max_iter=max_iter_pop)
sa = SA(max_iter=max_iter_single, T0=100, alpha=0.995)
hc = HillClimbing(max_iter=max_iter_single)

# Output directory
save_dir = "output"
os.makedirs(save_dir, exist_ok=True)

# Benchmark class to run the benchmark
runner = BenchmarkRunner(
    n_trials=n_trials,
    dim=dim,
    max_iter=max_iter_pop,
    seed=seed
)

CONTINUOUS OPTIMIZATION BENCHMARKS

In [3]:
# Define continuos Functions
CONTINUOUS_FUNCTIONS = {
    "Sphere":     (sphere,     (-100, 100)),
    "Rastrigin":  (rastrigin,  (-5.12, 5.12)),
    "Rosenbrock": (rosenbrock, (-5, 10)),
    "Griewank":   (griewank,   (-600, 600)),
    "Ackley":     (ackley,     (-32.768, 32.768)),
}

# Choosing algorithms to test continuous problems
continuous_algorithms = {
    "CS": cs,
    "FA": fa,
    "PSO": pso,
    "HC": hc,
    "SA": sa,
    "TLBO": tlbo,
    "DE": de
}

In [9]:
# RUNNING THE BENCHMARKS
total_t0 = time.perf_counter()
cont_results = runner.run_continuous_benchmarks(CONTINUOUS_FUNCTIONS, continuous_algorithms, verbose=True)
elapsed = time.perf_counter() - total_t0
print(f"\n{'=' * 60}")
print(f"  DONE  — total time {elapsed:.1f}s")
print(f"{'=' * 60}\n")

save_continuous_benchmarks(cont_results, save_dir="output")
export_continuous_csv(cont_results, save_dir = "output")

        Sphere | CS | mean=1.0359e+03  std=5.8486e+02
        Sphere | FA | mean=5.8568e+03  std=1.2352e+03
        Sphere | PSO | mean=3.9218e-45  std=8.1901e-45
        Sphere | HC | mean=8.5940e+03  std=4.8086e+03
        Sphere | SA | mean=2.8509e+04  std=5.3674e+03
        Sphere | TLBO | mean=1.8383e-228  std=0.0000e+00
        Sphere | DE | mean=1.4176e-12  std=1.5856e-12
     Rastrigin | CS | mean=1.0863e+01  std=4.6246e+00
     Rastrigin | FA | mean=1.6245e+01  std=8.2965e+00
     Rastrigin | PSO | mean=1.0248e+01  std=4.3381e+00
     Rastrigin | HC | mean=1.3054e+02  std=2.5062e+01
     Rastrigin | SA | mean=1.6664e+02  std=2.3906e+01
     Rastrigin | TLBO | mean=4.2792e+00  std=2.5193e+00
     Rastrigin | DE | mean=1.6298e+01  std=1.2013e+01
    Rosenbrock | CS | mean=5.6672e+00  std=3.0166e+00
    Rosenbrock | FA | mean=3.3775e+03  std=5.6482e+03
    Rosenbrock | PSO | mean=1.0277e+00  std=1.1665e+00
    Rosenbrock | HC | mean=2.7803e+04  std=2.4598e+04
    Rosenbrock | SA 

SCALABILITY RUNS 

In [10]:
dims = [10, 20, 30]
scalability = runner.run_scalability_benchmarks(dims, CONTINUOUS_FUNCTIONS, continuous_algorithms, verbose = True)
save_scalability_benchmarks(scalability, save_dir="output")


  Dimension = 10
        Sphere | CS | mean=1.0359e+03  std=5.8486e+02
        Sphere | FA | mean=5.8568e+03  std=1.2352e+03
        Sphere | PSO | mean=3.9218e-45  std=8.1901e-45
        Sphere | HC | mean=8.5940e+03  std=4.8086e+03
        Sphere | SA | mean=2.8509e+04  std=5.3674e+03
        Sphere | TLBO | mean=3.0896e-226  std=0.0000e+00
        Sphere | DE | mean=1.4176e-12  std=1.5856e-12
     Rastrigin | CS | mean=1.0863e+01  std=4.6246e+00
     Rastrigin | FA | mean=1.6245e+01  std=8.2965e+00
     Rastrigin | PSO | mean=1.0248e+01  std=4.3381e+00
     Rastrigin | HC | mean=1.3054e+02  std=2.5062e+01
     Rastrigin | SA | mean=1.6664e+02  std=2.3906e+01
     Rastrigin | TLBO | mean=2.6158e+00  std=1.2719e+00
     Rastrigin | DE | mean=1.6298e+01  std=1.2013e+01
    Rosenbrock | CS | mean=5.6672e+00  std=3.0166e+00
    Rosenbrock | FA | mean=3.3775e+03  std=5.6482e+03
    Rosenbrock | PSO | mean=1.0277e+00  std=1.1665e+00
    Rosenbrock | HC | mean=2.7803e+04  std=2.4598e+04
  

'/Users/thaihoang78/Project/Nature_Inspired_Algorithms/output/scalability_results.json'

EXPLORATON AND EXPLOITATION RUNS

In [11]:
# Exploration and Exploitation
results = runner.bench_exploration_exploitation(CONTINUOUS_FUNCTIONS, continuous_algorithms, verbose=True)
save_exploration_exploitation_benchmarks(results, save_dir="output")

        Sphere | CS           | avg_len=1000.0
        Sphere | FA           | avg_len=1000.0
        Sphere | PSO          | avg_len=1000.0
        Sphere | HC           | avg_len=6.2
        Sphere | SA           | avg_len=11.0
        Sphere | TLBO         | avg_len=1000.0
        Sphere | DE           | avg_len=1000.0
     Rastrigin | CS           | avg_len=1000.0
     Rastrigin | FA           | avg_len=1000.0
     Rastrigin | PSO          | avg_len=1000.0
     Rastrigin | HC           | avg_len=2.4
     Rastrigin | SA           | avg_len=11.0
     Rastrigin | TLBO         | avg_len=1000.0
     Rastrigin | DE           | avg_len=1000.0
    Rosenbrock | CS           | avg_len=1000.0
    Rosenbrock | FA           | avg_len=1000.0
    Rosenbrock | PSO          | avg_len=1000.0
    Rosenbrock | HC           | avg_len=8.1
    Rosenbrock | SA           | avg_len=11.0
    Rosenbrock | TLBO         | avg_len=1000.0
    Rosenbrock | DE           | avg_len=1000.0
      Griewank | CS         

'/Users/thaihoang78/Project/Nature_Inspired_Algorithms/output/exploration_exploitation_results.json'

DISCRETE OPTIMIZATION BENCHMARKS

In [4]:
# Initialize 3 discrete problems: Traveling Sales Problem, Knapsack Problem, Graph Colouring Problem
tsp = TSP(np.zeros((1,1))) # dummy init
gc = GraphColoring(1)   # temporary init for n_vertices 
kp = KnapsackProblem([1], [1], 1)  # dummy init

# Chosing prepared testcases for experiments
tsp.load_from_json("test_case/small_tsp_10.json")
gc.load_from_json("test_case/gc_small_sparse.json")
kp.load_from_json("test_case/knapsack_small.json")

In [6]:
# Create problem solver classes of each problems
TSP_solver = TSPSolver(tsp)
Knapsack_solver = KnapsackSolver(kp)
Graphcoloring_solver = GraphColoringSolver(gc, n_colors = gc.n_vertices)

# Define specific algorithms to solve problems
TSP_algos = {
    "SA (TSP)": lambda: TSP_solver.solve_sa(T0=500.0, T_min=1e-3, max_iter=max_iter_single, alpha=0.001, verbose=False),
    "GA (TSP)": lambda: TSP_solver.solve_ga(pop_size=pop_size, max_iter=max_iter_pop, CR=0.9, mutation_rate=0.02, tournament_size=3, elitism_rate=0.1, beta=2.0, verbose=False),
    "ACO (TSP)": lambda: TSP_solver.solve_aco(n_ants=pop_size, max_iter=max_iter_pop, alpha=1.0, beta_aco=3.0, rho=0.1, Q=100.0, tau_init=1.0, verbose=False),
    "CS (TSP)": lambda: TSP_solver.solve_cs(n_nests=pop_size, max_iter=max_iter_pop, pa=0.25, verbose=False),
    "ABC (TSP)": lambda: TSP_solver.solve_abc(n_bees=pop_size, max_iter=max_iter_pop, limit=pop_size * dim, verbose=False),
    "FA (TSP)": lambda: TSP_solver.solve_fa(n_fireflies=pop_size, max_iter=max_iter_pop, alpha=0.5, beta0=1.0, gamma=0.1, alpha_decay=0.97, verbose=False),
    "A* (TSP)": lambda: TSP_solver.solve_astar(verbose=False),
    "BFS (TSP)": lambda: TSP_solver.solve_bfs(beam_width=10, verbose=False),
    "DFS (TSP)": lambda: TSP_solver.solve_dfs(max_nodes=50_000, verbose=False),
}

Knapsack_algos = {
    "SA (Knapsack)": lambda: Knapsack_solver.solve_sa(max_iter=max_iter_single, verbose=False),
    "GA (Knapsack)": lambda: Knapsack_solver.solve_ga(pop_size=pop_size, max_iter=max_iter_pop, verbose=False),
    "ACO (Knapsack)": lambda: Knapsack_solver.solve_aco(n_ants=pop_size, max_iter=max_iter_pop, verbose=False),
    "CS (Knapsack)": lambda: Knapsack_solver.solve_cs(n_nests=pop_size, max_iter=max_iter_pop, verbose=False),
    "ABC (Knapsack)": lambda: Knapsack_solver.solve_abc(n_bees=pop_size, max_iter=max_iter_pop, verbose=False),
    "FA (Knapsack)": lambda: Knapsack_solver.solve_fa(n_fireflies=pop_size, max_iter=max_iter_pop, verbose=False),
    "A* (Knapsack)": lambda: Knapsack_solver.solve_astar(verbose=False),
    "BFS (Knapsack)": lambda: Knapsack_solver.solve_bfs(beam_width=20, verbose=False),
    "DFS (Knapsack)": lambda: Knapsack_solver.solve_dfs(max_nodes=100_000, verbose=False),
}

Graph_colouring_algos = {
    "SA (GC)": lambda: Graphcoloring_solver.solve_sa(max_iter=max_iter_single, verbose=False),
    "GA (GC)": lambda: Graphcoloring_solver.solve_ga(pop_size=pop_size, max_iter=max_iter_pop, verbose=False),
    "ACO (GC)": lambda: Graphcoloring_solver.solve_aco(n_ants=pop_size, max_iter=max_iter_pop, verbose=False),
    "CS (GC)": lambda: Graphcoloring_solver.solve_cs(n_nests=pop_size, max_iter=max_iter_pop, verbose=False),
    "ABC (GC)": lambda: Graphcoloring_solver.solve_abc(n_bees=pop_size, max_iter=max_iter_pop, verbose=False),
    "FA (GC)": lambda: Graphcoloring_solver.solve_fa(n_fireflies=pop_size, max_iter=max_iter_pop, verbose=False),
    "A* (GC)": lambda: Graphcoloring_solver.solve_astar(verbose=False),
    "BFS (GC)": lambda: Graphcoloring_solver.solve_bfs(beam_width=20, verbose=False),
    "DFS (GC)": lambda: Graphcoloring_solver.solve_dfs(max_nodes=10_000, verbose=False),
}

# RUNNING THE BENCHMARKS
print("\n" + "=" * 60)
print("  DISCRETE PROBLEM BENCHMARKS")
print("=" * 60)

# Benmark the algorithms based on those discrete problems

total_t0 = time.perf_counter()

disc_results = {}
disc_results["TSP"] = runner._bench_tsp(TSP_algos, tsp, verbose = True)
disc_results["Knapsack"] = runner._bench_knapsack(Knapsack_algos, kp, verbose = True)
disc_results["Graph Coloring"] = runner._bench_graph_coloring(Graph_colouring_algos, gc, verbose = True)

elapsed = time.perf_counter() - total_t0

print(f"\n{'=' * 60}")
print(f"  DONE  — total time {elapsed:.1f}s")
print(f"{'=' * 60}\n")

save_discrete_benchmarks(disc_results, save_dir)


  DISCRETE PROBLEM BENCHMARKS
  TSP | SA (TSP)             | mean=290.31
  TSP | GA (TSP)             | mean=295.13
  TSP | ACO (TSP)            | mean=290.31
  TSP | CS (TSP)             | mean=290.31
  TSP | ABC (TSP)            | mean=290.31
  TSP | FA (TSP)             | mean=302.48
  TSP | A* (TSP)             | mean=290.31
  TSP | BFS (TSP)            | mean=315.81
  TSP | DFS (TSP)            | mean=290.31
  Knapsack | SA (Knapsack)        | mean=831.90
  Knapsack | GA (Knapsack)        | mean=796.00
  Knapsack | ACO (Knapsack)       | mean=869.70
  Knapsack | CS (Knapsack)        | mean=871.00
  Knapsack | ABC (Knapsack)       | mean=818.70
  Knapsack | FA (Knapsack)        | mean=871.00
  Knapsack | A* (Knapsack)        | mean=871.00
  Knapsack | BFS (Knapsack)       | mean=871.00
  Knapsack | DFS (Knapsack)       | mean=871.00
  Graph Coloring | SA (GC)              | mean=3.00
  Graph Coloring | GA (GC)              | mean=3.20
  Graph Coloring | ACO (GC)             | mean

TypeError: 'TSP' object is not iterable